# Can uncertainty identify incorrect predictions?

This notebook compares MC Dropout, Deep Ensemble, Training Subsample Ensemble, Last-Layer Laplace Approximation, and TabICLv2. All methods receive the same controlled feature configuration: XYZ, depth, and PCA16 text embeddings.

The main question concerns **error ranking**, not calibration alone: do uncertain predictions actually contain more errors?

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import bovino_results as br

## 1. Five mechanisms, one common probability decomposition

Each method produces multiple plausible class-probability vectors $p_m$. Their origins differ, but the same summaries can be calculated:

$$H[\bar p] = -\sum_c \bar p_c\log \bar p_c,$$

$$\mathbb{E}[H[p_m]] = -\frac{1}{M}\sum_m\sum_c p_{m,c}\log p_{m,c},$$

$$MI = H[\bar p]-\mathbb{E}[H[p_m]].$$

Predictive entropy measures total ambiguity. Mutual information measures disagreement among the probability vectors and acts as an epistemic proxy.

## 2. Accuracy and uncertainty answer different questions

Overall accuracy and Cohen's kappa evaluate the predicted labels. Error AUROC evaluates whether an uncertainty score ranks incorrect predictions above correct ones. A value of 0.5 corresponds to random ranking.

Confidence, predictive entropy, and MI can therefore be compared even when their raw numerical scales differ.

In [ ]:
display(br.uq_benchmark_summary('lobo').round(3))
display(br.uq_benchmark_summary('loco').round(3))

## 3. Error detection weakens under unseen campaigns

In LOBO, confidence and predictive entropy generally produce Error AUROC values around 0.75 to 0.77. Under LOCO they fall to roughly 0.66 to 0.71. MI is usually weaker and no method dominates every score and protocol.

In [ ]:
fig, axes = br.plot_uq_benchmark()
plt.show()

## 4. Calibration is a separate property

Expected Calibration Error compares confidence with empirical accuracy. Post-hoc calibration can improve ECE without changing which examples the model ranks as risky. For selective review or abstention, Error AUROC, Error AUPRC, and risk-coverage curves remain necessary.

In [ ]:
fig, axes = br.plot_loco_reliability_and_roc()
plt.show()

## Take-home message

All five methods provide useful but moderate error detection. The stronger campaign shift reduces their discrimination, and method rankings depend on the chosen score. Confidence remains a strong baseline; a more elaborate epistemic proxy does not automatically produce safer decisions.